# Experimental quantized Urdu CPT

Alternative 4-bit Llama 3.1 8B configuration. This notebook is not the primary paper-aligned implementation and is not evidence of the published run.

Use the environment in `requirements.txt` on Linux with an NVIDIA GPU. Run the setup cell first, then restart the kernel after installation. See `docs/training_details.md` for differences and limitations.


In [ ]:
from pathlib import Path
import os

repo_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "requirements.txt").is_file() and (p / "training/cpt_train.py").is_file()),
    None,
)
if repo_root is None:
    raise RuntimeError("Open the notebook from within the Qalb-1.0 repository.")
os.chdir(repo_root)
%pip install -r requirements.txt


In [ ]:
import unsloth

import os, math, gc, random
from dataclasses import dataclass
from typing import Optional, List, Tuple

import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, TrainerCallback
from transformers.trainer_utils import get_last_checkpoint
from huggingface_hub import login
import getpass

from unsloth import (
    FastLanguageModel,
    UnslothTrainer,
    UnslothTrainingArguments,
    is_bfloat16_supported,
)

# TF32 on A100
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

SEED = 3407
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)


In [ ]:
# Hugging Face login

HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    try:
        HF_TOKEN = getpass.getpass("Enter your Hugging Face token (starts with 'hf_'): ").strip()
    except Exception:
        HF_TOKEN = ""
if HF_TOKEN:
    try:
        login(token=HF_TOKEN, add_to_git_credential=False)
        os.environ["HF_TOKEN"] = HF_TOKEN
        print("✅ Hugging Face login successful.")
    except Exception as e:
        print(f"⚠️ HF login failed (continuing without): {e}")
else:
    print("ℹ️ No HF token provided. Will only use local artifacts or public models.")


In [ ]:
# Configuration

@dataclass
class Config:
    # Data
    bin_path: str = "llama31_8b_mixed_tokens/llama31_8b_mixed_seq2048.bin"  # Flat int32 tokens, packed into 2048-token rows.
    seq_len: int = 2048
    eval_ratio: float = 0.005
    shuffle_indices: bool = True

    # Tokenizer
    local_tokenizer_dir: str = "./local_tokenizer"
    hf_tokenizer_candidates: Tuple[str, ...] = (
        "unsloth/Meta-Llama-3.1-8B",
        "meta-llama/Meta-Llama-3.1-8B",
    )

    # 4-bit base model
    base_model_id: str = "unsloth/Meta-Llama-3.1-8B-bnb-4bit"

    # LoRA
    lora_r: int = 128
    lora_alpha: int = 32
    lora_dropout: float = 0.0
    target_modules: Tuple[str, ...] = (
        "q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj",
        "embed_tokens","lm_head",
    )
    bias: str = "none"

    # Optimization
    learning_rate: float = 2e-5
    embedding_learning_rate: float = 2e-6 # Lower LR for embeddings and lm_head.
    weight_decay: float = 0.0
    lr_scheduler_type: str = "cosine"
    warmup_ratio: float = 0.05

    # Batching and precision
    per_device_train_batch_size: int = 1
    gradient_accumulation_steps: int = 16
    per_device_eval_batch_size: int = 1
    bf16: bool = True
    fp16: bool = False

    # Training length
    num_train_epochs: float = 1.0
    # Setting max_steps overrides num_train_epochs.

    # Checkpoints and logging
    output_dir: str = "outputs_urdu_cpt"
    logging_steps: int = 50
    eval_steps: int = 1000
    save_steps: int = 1000
    save_total_limit: int = 2
    report_to: Optional[str] = "none"
    csv_log_path: str = "training_logs.csv"

cfg = Config()
cfg


In [ ]:
# Tokenizer

def load_tokenizer(cfg: Config) -> AutoTokenizer:
    tok_local = tok_hf = None

    if os.path.isdir(cfg.local_tokenizer_dir):
        print(f"Loading local tokenizer from: {cfg.local_tokenizer_dir}")
        tok_local = AutoTokenizer.from_pretrained(
            cfg.local_tokenizer_dir,
            use_fast=True,
            local_files_only=True,
        )

    if tok_local is None:
        for repo in cfg.hf_tokenizer_candidates:
            try:
                print(f"Trying HF tokenizer: {repo}")
                tok_hf = AutoTokenizer.from_pretrained(
                    repo,
                    use_fast=True,
                    token=os.environ.get("HF_TOKEN") or None,
                )
                print(f"Loaded HF tokenizer: {repo}")
                break
            except Exception as e:
                print(f"Failed: {repo} ({e})")
        if tok_hf is None:
            raise RuntimeError("❌ Could not load a Llama-3.1-8B tokenizer (local or HF).")

    tok = tok_local or tok_hf

    if tok.pad_token is None and tok.eos_token is not None:
        tok.pad_token = tok.eos_token
        tok.pad_token_id = tok.eos_token_id

    if tok_local is not None and tok_hf is not None:
        sample_text = "پاکستان میں آج موسم کیسا ہے؟"
        ids_local = tok_local(sample_text, add_special_tokens=False)["input_ids"]
        ids_hf    = tok_hf(sample_text, add_special_tokens=False)["input_ids"]
        if ids_local != ids_hf:
            print("⚠️ Local and HF tokenizer IDs differ on a sample. "
                  "For safety, we will use LOCAL (matching your bin packing).")
        else:
            print("✅ Local and HF tokenizer IDs match on sample text.")

    return tok

tokenizer = load_tokenizer(cfg)
print("BOS/EOS IDs:", tokenizer.bos_token_id, tokenizer.eos_token_id, "| Vocab size:", tokenizer.vocab_size)


In [ ]:
def validate_packed_tokens(cfg, tokenizer):
    """Check file layout and token IDs without modifying the corpus."""
    size = os.path.getsize(cfg.bin_path)
    row_bytes = np.dtype(np.int32).itemsize * cfg.seq_len
    if size % row_bytes or size < 2 * row_bytes:
        raise ValueError("Packed data must contain at least two complete int32 token rows.")
    valid_ids = set(tokenizer.get_vocab().values())
    if not valid_ids or min(valid_ids) < 0:
        raise ValueError("Tokenizer vocabulary must contain non-negative token IDs.")
    # Check all IDs in bounded chunks; no token replacement or filtering.
    max_id = max(valid_ids)
    allowed = np.zeros(max_id + 1, dtype=np.bool_)
    allowed[list(valid_ids)] = True
    packed = np.memmap(cfg.bin_path, dtype=np.int32, mode="r")
    for start in range(0, packed.size, 1_048_576):
        chunk = packed[start:start + 1_048_576]
        if chunk.min() < 0 or chunk.max() > max_id or not allowed[chunk].all():
            raise ValueError("Packed data contains token IDs outside the tokenizer vocabulary.")
    del packed


def validate_tokenizer_model(tokenizer, model_tokenizer, model):
    """Reject incompatible token mappings before adapter attachment."""
    vocab = tokenizer.get_vocab()
    if vocab != model_tokenizer.get_vocab():
        raise ValueError("Local and model token-to-ID mappings differ. Use the tokenizer used for packing.")
    for name in ("bos_token_id", "eos_token_id"):
        if getattr(tokenizer, name) != getattr(model_tokenizer, name):
            raise ValueError(f"Tokenizer {name} differs from the model tokenizer.")
    input_rows = model.get_input_embeddings().weight.shape[0]
    output_rows = model.get_output_embeddings().weight.shape[0]
    ids = list(vocab.values()) + list(tokenizer.all_special_ids)
    if not ids or min(ids) < 0 or max(ids) >= min(input_rows, output_rows):
        raise ValueError("Tokenizer IDs exceed the model embedding or output vocabulary.")
    print("Tokenizer mappings and embedding bounds checked; no vocabulary resizing performed.")


# Packed token dataset

class TokenBinDataset(Dataset):
    def __init__(self, bin_path: str, seq_len: int, rows: Optional[List[int]] = None):
        self.bin_path = bin_path
        self.seq_len = seq_len
        self.mm = np.memmap(bin_path, dtype=np.int32, mode="r")
        if self.mm.size % seq_len != 0:
            raise ValueError(f"Bin size {self.mm.size} not divisible by seq_len {seq_len}.")
        self.n_rows = self.mm.size // seq_len
        self.rows = rows if rows is not None else list(range(self.n_rows))

    def __len__(self):
        return len(self.rows)

    def map(self, function, batched=False, batch_size=1000, desc=None, **kwargs):
        """Apply a batched side-effect transform used by Unsloth's token checks."""
        if not batched:
            raise NotImplementedError("TokenBinDataset.map supports batched transforms only.")
        if batch_size < 1:
            raise ValueError("batch_size must be positive.")
        for start in range(0, len(self), batch_size):
            stop = min(start + batch_size, len(self))
            rows = []
            for index in range(start, stop):
                row = self.rows[index]
                offset = row * self.seq_len
                rows.append(self.mm[offset:offset + self.seq_len].tolist())
            function({"input_ids": rows})
        return self

    def __getitem__(self, idx):
        r = self.rows[idx]
        start = r * self.seq_len
        end = start + self.seq_len
        ids = np.array(self.mm[start:end], dtype=np.int64)
        return {
            "input_ids": torch.from_numpy(ids),
            "labels": torch.from_numpy(ids.copy()),
            "attention_mask": torch.ones(self.seq_len, dtype=torch.long),
        }

def build_train_eval_from_bin(cfg: Config):
    assert os.path.isfile(cfg.bin_path), f"❌ Packed .bin not found at {cfg.bin_path}"
    validate_packed_tokens(cfg, tokenizer)
    mm = np.memmap(cfg.bin_path, dtype=np.int32, mode="r")
    n_rows_total = mm.size // cfg.seq_len
    del mm; gc.collect()

    idx = np.arange(n_rows_total)
    if cfg.shuffle_indices:
        rng = np.random.default_rng(SEED)
        rng.shuffle(idx)

    n_eval = max(1, int(n_rows_total * cfg.eval_ratio))
    eval_rows = idx[:n_eval].tolist()
    train_rows = idx[n_eval:].tolist()

    train_ds = TokenBinDataset(cfg.bin_path, cfg.seq_len, train_rows)
    eval_ds  = TokenBinDataset(cfg.bin_path, cfg.seq_len, eval_rows)

    tokens_per_epoch = len(train_rows) * cfg.seq_len
    print(f"Total rows: {n_rows_total:,} | Train rows: {len(train_rows):,} | Eval rows: {len(eval_rows):,}")
    print(f"Tokens per epoch (train only): {tokens_per_epoch:,}")

    bos_id = tokenizer.bos_token_id
    sample = min(200, len(train_ds))
    if bos_id is not None and sample > 0:
        hits = 0
        for i in range(sample):
            if train_ds[i]["input_ids"][0].item() == bos_id:
                hits += 1
        print(f"BOS@0: {hits}/{sample} train rows start with BOS (expected: high).")

    return train_ds, eval_ds

train_ds, eval_ds = build_train_eval_from_bin(cfg)


In [ ]:
# Decode sample

row0 = eval_ds[0]["input_ids"].tolist()
preview = tokenizer.decode(row0[:128], skip_special_tokens=False)
print(preview.replace("\n", " ")[:400], "…")


In [ ]:
# Base model and LoRA

model_kwargs = {}
if os.environ.get("HF_TOKEN"):
    model_kwargs["token"] = os.environ["HF_TOKEN"]

model, _tok_from_model = FastLanguageModel.from_pretrained(
    model_name = cfg.base_model_id,
    max_seq_length = cfg.seq_len,
    dtype = None,
    load_in_4bit = True,
    **model_kwargs,
)

if tokenizer.pad_token is None and tokenizer.eos_token is not None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

validate_tokenizer_model(tokenizer, _tok_from_model, model)

try:
    print("Model vocab_size:", getattr(model.config, "vocab_size", None),
          "| Tokenizer vocab_size:", tokenizer.vocab_size)
except:
    pass

model = FastLanguageModel.get_peft_model(
    model,
    r = cfg.lora_r,
    target_modules = list(cfg.target_modules),
    lora_alpha = cfg.lora_alpha,
    lora_dropout = cfg.lora_dropout,
    bias = cfg.bias,
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
    max_seq_length = cfg.seq_len,
)


In [ ]:
# Fixed-length collator

def collate_fixed_rows(batch):
    input_ids = torch.stack([b["input_ids"] for b in batch])
    labels = torch.stack([b["labels"] for b in batch])
    attention_mask = torch.stack([b["attention_mask"] for b in batch])
    return {"input_ids": input_ids, "labels": labels, "attention_mask": attention_mask}


In [ ]:
# CSV logging

class CSVLoggerCallback(TrainerCallback):
    def __init__(self, path: str):
        self.path = path
        self._f = None

    def _open(self):
        if self._f is None:
            os.makedirs(os.path.dirname(self.path) or ".", exist_ok=True)
            self._f = open(self.path, "w", encoding="utf-8")
            self._f.write("step,loss,eval_loss,eval_perplexity\n")
            self._f.flush()

    def on_train_begin(self, args, state, control, **kwargs):
        self._open()

    def on_log(self, args, state, control, logs=None, **kwargs):
        self._open()
        logs = logs or {}
        step = state.global_step
        loss = logs.get("loss")
        eval_loss = logs.get("eval_loss")
        ppl = math.exp(eval_loss) if eval_loss is not None else None
        line = f"{step},{'' if loss is None else loss},{'' if eval_loss is None else eval_loss},{'' if ppl is None else ppl}\n"
        self._f.write(line); self._f.flush()

    def on_train_end(self, args, state, control, **kwargs):
        if self._f:
            self._f.close()
            self._f = None


In [ ]:
# Training arguments

training_args = UnslothTrainingArguments(
    output_dir = cfg.output_dir,
    per_device_train_batch_size = cfg.per_device_train_batch_size,
    per_device_eval_batch_size  = cfg.per_device_eval_batch_size,
    gradient_accumulation_steps = cfg.gradient_accumulation_steps,
    learning_rate = cfg.learning_rate,
    embedding_learning_rate = cfg.embedding_learning_rate,
    weight_decay = cfg.weight_decay,
    lr_scheduler_type = cfg.lr_scheduler_type,
    warmup_ratio = cfg.warmup_ratio,
    num_train_epochs = cfg.num_train_epochs,
    logging_steps = cfg.logging_steps,
    eval_strategy = "steps",
    eval_steps = cfg.eval_steps,
    save_strategy = "steps",
    save_steps = cfg.save_steps,
    save_total_limit = cfg.save_total_limit,
    bf16 = cfg.bf16 if is_bfloat16_supported() else False,
    fp16 = cfg.fp16 and not is_bfloat16_supported(),
    optim = "adamw_8bit",
    report_to = cfg.report_to,
    seed = SEED,
    max_seq_length = cfg.seq_len,
    packing = False,
    dataset_kwargs = {"skip_prepare_dataset": True},
)

training_args


In [ ]:
# Trainer

trainer = UnslothTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = eval_ds,
    data_collator = collate_fixed_rows,
    args = training_args,
)
# Preserve the embedding LR if TRL rebuilds the argument object.
trainer.args.embedding_learning_rate = cfg.embedding_learning_rate
trainer.add_callback(CSVLoggerCallback(cfg.csv_log_path))


In [ ]:
# Resume training

last_checkpoint = get_last_checkpoint(cfg.output_dir) if os.path.isdir(cfg.output_dir) else None
train_result = trainer.train(resume_from_checkpoint=last_checkpoint)
train_result


In [ ]:
# Save adapters

adapters_dir = "urdu_cpt_lora"
os.makedirs(adapters_dir, exist_ok=True)
model.save_pretrained(adapters_dir)
tokenizer.save_pretrained(adapters_dir)
print(f"✅ LoRA adapters saved to: {adapters_dir}")


In [ ]:
# Final evaluation

metrics = trainer.evaluate()
eval_loss = metrics.get("eval_loss")
if eval_loss is not None:
    ppl = math.exp(eval_loss)
    print(f"Final eval loss: {eval_loss:.4f} | Perplexity: {ppl:.2f}")
    with open(cfg.csv_log_path, "a", encoding="utf-8") as f:
        f.write(f"{trainer.state.global_step},,{eval_loss},{ppl}\n")
else:
    print("⚠️ No eval_loss reported — check eval split / eval_ratio.")


In [ ]:
# Generation sample

from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

prompt = "پاکستان میں آج کی اہم خبروں کی سرخیوں کا خلاصہ پیش کریں:"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=True,
        temperature=0.8,
        top_p=0.9,
    )

print(tokenizer.decode(out[0], skip_special_tokens=True))
